# Solaris RL — Kaggle Training Notebook

Resumable DQN / Double DQN training for Atari Solaris.

**Kaggle session limit: 9 hours, hard cutoff.** This notebook is designed to be
re-run across multiple sessions — it auto-resumes from the last saved checkpoint
each time, using Kaggle's persistent **Output** directory to carry checkpoints
between sessions.

### How to use this notebook across multiple sessions
1. **First run:** Settings (right sidebar) → Accelerator → **GPU T4 x2**. Run all cells.
2. When the 9-hour limit hits (or you stop it manually), the notebook stops —
   but checkpoints are already saved in `/kaggle/working/checkpoints/`.
3. **To continue:** open this same notebook again (it auto-saves your edits),
   click **Save Version → Save & Run All**. Kaggle restores your previous
   `/kaggle/working/` output automatically, so checkpoints persist and
   training resumes exactly where it left off.
4. Repeat until `total_steps` is reached.

### Before you start
- Push your `solaris_rl` project to a **public (or Kaggle-accessible private) GitHub repo** first.
- Set `REPO_URL` in the cell below.
- (Optional but recommended) Add your W&B API key as a Kaggle **Secret** named `WANDB_API_KEY`
  so you can monitor training remotely from your phone instead of watching this notebook.

In [ ]:
# ============================================================
# CONFIG — edit these before running
# ============================================================
REPO_URL = "https://github.com/<your-username>/atari-solaris.git"  # <-- EDIT THIS
BRANCH = "main"

# Which experiment config to run. Must match a file in configs/.
CONFIG_PATH = "configs/double_dqn.yaml"
SEED = 616

# Set to True once you've added a WANDB_API_KEY Kaggle Secret
USE_WANDB = True

In [ ]:
# ============================================================
# 1. Clone (or update) the repo into /kaggle/working
# ============================================================
# /kaggle/working persists across sessions for the SAME notebook, so we clone
# once and `git pull` on subsequent runs rather than re-cloning every time.
import os

PROJECT_DIR = "/kaggle/working/atari-solaris"

if not os.path.isdir(PROJECT_DIR):
    print("Cloning repo (first run)...")
    !git clone --branch {BRANCH} {REPO_URL} {PROJECT_DIR}
else:
    print("Repo already present — pulling latest changes...")
    !cd {PROJECT_DIR} && git pull

os.chdir(PROJECT_DIR)
print("\nWorking directory:", os.getcwd())

In [ ]:
# ============================================================
# 2. Install dependencies
# ============================================================
# Kaggle images already ship PyTorch + CUDA, so we skip reinstalling torch
# (reinstalling it tends to break the preconfigured CUDA linkage).
!pip install -q ale-py gymnasium[accept-rom-license] AutoROM opencv-python-headless wandb pyyaml
!AutoROM --accept-license -y > /dev/null 2>&1
print("✅ Dependencies installed")

In [ ]:
# ============================================================
# 3. Verify GPU and environment
# ============================================================
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

import gymnasium as gym
import ale_py
gym.register_envs(ale_py)
env = gym.make("ALE/Solaris-v5", frameskip=1)
obs, _ = env.reset()
print("✅ Solaris env OK — obs shape:", obs.shape)
env.close()

In [ ]:
# ============================================================
# 4. (Optional) Log in to Weights & Biases for remote monitoring
# ============================================================
# Add your key via: Notebook → Add-ons → Secrets → WANDB_API_KEY
# train.py runs as a subprocess (!python ...), so we must set the
# WANDB_API_KEY environment variable — not just call wandb.login() —
# for the child process to authenticate automatically.
import os
if USE_WANDB:
    from kaggle_secrets import UserSecretsClient
    try:
        wandb_key = UserSecretsClient().get_secret("WANDB_API_KEY")
        os.environ["WANDB_API_KEY"] = wandb_key
        import wandb
        wandb.login(key=wandb_key, relogin=True)
        print("✅ W&B logged in — monitor this run from wandb.ai on your phone")
    except Exception as e:
        print(f"⚠️  W&B login skipped ({e}); add a WANDB_API_KEY secret to enable.")
        USE_WANDB = False

In [ ]:
# ============================================================
# 5. Symlink checkpoints into /kaggle/working so they PERSIST
#    across sessions (this is the critical resumability step)
# ============================================================
# Everything under /kaggle/working is saved as this notebook's "Output" and
# restored automatically the next time you open and run this same notebook.
# checkpoints/ already lives inside PROJECT_DIR, which is itself inside
# /kaggle/working, so no extra symlinking is actually needed — but we assert
# it here explicitly so it's obvious and verifiable.
ckpt_root = os.path.join(PROJECT_DIR, "checkpoints")
os.makedirs(ckpt_root, exist_ok=True)
assert ckpt_root.startswith("/kaggle/working"), "Checkpoints must live under /kaggle/working to persist!"
print("✅ Checkpoints will persist at:", ckpt_root)

# Show what's already there (non-empty on a resumed session)
for root, dirs, files in os.walk(ckpt_root):
    for f in files:
        print(" -", os.path.join(root, f))

In [ ]:
# ============================================================
# 6. Train — auto-resumes if a checkpoint already exists
# ============================================================
# --resume is always passed:
#   - No checkpoint found → train.py starts fresh, W&B run wiped clean (force=True)
#   - Checkpoint found    → weights/optimiser/step restored, W&B appends to same run
# Either way, no 'step must be monotonically increasing' warnings.
wandb_flag = "" if USE_WANDB else "--no-wandb"

!python train.py --config {CONFIG_PATH} --seed {SEED} --resume {wandb_flag}

---
### What happens when this session times out mid-training

Nothing bad — that's the point. The training loop checkpoints every
`checkpoint_freq` steps (100K by default) to `checkpoints/<exp_name>/`.
When you come back and re-run this notebook:

- The repo is already cloned (just `git pull`s for any code updates).
- `train.py --resume` finds the latest `step_*.pt` checkpoint and restores
  network weights, optimiser state, and the step counter.
- The replay buffer starts empty and refills over the first `learning_starts`
  steps — a small, expected cost, not a bug.
- If W&B is enabled, the same run ID is reused so your learning curve on
  wandb.ai is one continuous line across sessions, not several disconnected ones.

### Estimating sessions needed
At ~150–300 SPS on a Kaggle T4 (variable with load), a 10M-step run needs
roughly **10–18 hours of GPU time total** → **2–3 Kaggle sessions** at the
9-hour cap. Kaggle gives 30 free GPU-hours/week, which covers this easily.